# M11 on Kaggle: LLM verification of E4 (`e4_added`, D11), E5 and E6, on the full pool

**Seed:** set `SEED` in the first code cell. Run 42 first, then 43 in a second session. M11 reads M9's report for the same seed (K, gate strength, scores to check against), which is in the repo.

**Attach as input:**
1. The same private Kaggle dataset as `m3_kaggle.ipynb` (the five files of the M1 `processed/` folder).
2. The model: *Add Input → Models →* **Gemma 2**, framework *Transformers*, variation **gemma-2-2b-it** (accept Google's licence on the model page first). The notebook finds it under `/kaggle/input`. Without it, it downloads `google/gemma-2-2b-it` from Hugging Face, which needs a Kaggle secret `HF_TOKEN` from an account that accepted the licence.

**Settings:** Internet *On*. Accelerator: **GPU T4** (one is enough).
**Run:** *Save Version → Save & Run All (Commit)*.

Steps: a GPU smoke test on 8 prompts (stops early if the model can't load or replies badly), then `fit` (CPU: reruns `e4_added`, about 80 min in M9), then `llm` (GPU). The smoke test prints a time estimate for `llm`; the full time is not measured yet. Every step prints an exit code: 137 means the process was killed (usually out of memory).

**Output:** `/kaggle/working/m11_full_seed42_outputs.zip` (the name follows `SEED`) contains `reports/m11/full/` (with every prompt and reply in `llm_log.jsonl`) and the run folders. Download it and bring it back, also after a failure.

In [ ]:
SEED = 42
TAG = "" if SEED == 42 else f"seed{SEED}"   # seed 42 is the main report, as in M7
TAG_ARG = f"--tag {TAG}" if TAG else ""
import os, subprocess, sys
REPO = "/kaggle/working/IDS-code"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Indrajit147/IDS-code.git", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
# --no-deps: keep Kaggle's preinstalled stack intact; versions are recorded per run. bitsandbytes isn't preinstalled.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO, "--no-deps"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes"], check=True)
os.chdir(REPO)
sys.path.insert(0, f"{REPO}/src")   # this kernel started before the editable install, so it cannot see it
os.environ["IDS_REPO_ROOT"] = REPO
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
assert os.path.exists("scripts/m11_verify.py"), "This clone predates M11 (no scripts/m11_verify.py). Restart the session so it clones the latest main."

In [ ]:
# The model: a Kaggle model input if attached, else Hugging Face (needs the HF_TOKEN secret)
import glob
found = sorted(os.path.dirname(p) for p in glob.glob("/kaggle/input/**/config.json", recursive=True) if "gemma-2-2b-it" in p.lower())
if found:
    MODEL = found[-1]
else:
    MODEL = "google/gemma-2-2b-it"
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
print("model:", MODEL)
LLM_SET = f"llm.backend=hf llm.model_id={MODEL}"

In [ ]:
# GPU smoke test: load the model, verify 8 made-up rows, check the replies and time them. Stops here on failure.
import json, time, torch
assert torch.cuda.is_available(), "No GPU: set Accelerator to GPU T4"
from ids.data.schema import CATEGORIES
from ids.llm.verifier import HFBackend, prompt, verify
be = HFBackend(MODEL, "4bit", "float16", 256)
pay = {"edge_label": "dos", "probs": {c: round(v, 3) for c, v in zip(CATEGORIES, [0.02, 0.35, 0.55, 0.04, 0.02, 0.02])},
       "booster_labels": {"xgboost": "dos", "lightgbm": "ddos"}, "disagreement": 0.31,
       "top_features": [{"feature": "Rate", "value": 2210.0, "shap": 1.4}, {"feature": "Number", "value": 9.5, "shap": 0.6}, {"feature": "Header_Length", "value": 54.0, "shap": -0.3}],
       "class_medians": {"Rate": {"dos": 2500.0, "ddos": 8000.0, "benign": 12.0}, "Number": {"dos": 9.5, "ddos": 9.5, "benign": 5.5}, "Header_Length": {"dos": 54.0, "ddos": 54.0, "benign": 300.0}},
       "recent_class_share": {c: v for c, v in zip(CATEGORIES, [0.03, 0.4, 0.5, 0.07, 0.0, 0.0])}, "drift": {"flag": True, "score": 0.12}, "known_classes": ["benign", "ddos", "dos", "recon"]}
feats = ["Rate", "Number", "Header_Length"]
tic = time.perf_counter()
recs, batches = verify([{"key": i, "payload": pay} for i in range(8)], be, CATEGORIES, feats, batch_size=8, max_retries=2)
sec = time.perf_counter() - tic
print(be.generate([prompt(pay, CATEGORIES)])[0])
print("valid:", sum(r["valid"] for r in recs), "of 8; first try:", sum(r["first_try_valid"] for r in recs), f"; {sec:.1f} s for 8 rows")
print(f"rough estimate: {1000 * sec / 8 / 60:.0f} min per 1,000 rows verified (the llm step logs how many rows it has)")
del be; torch.cuda.empty_cache()
assert sum(r["valid"] for r in recs) >= 6, "The model's replies mostly fail the contract: stop and bring the output back"

In [ ]:
# Rebuild the M2 partition for the full pool (deterministic, ~1 min)
!python scripts/m2_build_partitions.py --env kaggle --pools full
# M2 redraws these figures, and their bytes differ on Kaggle. Nothing reads them. A changed report or JSON stays flagged.
!git checkout -- 'reports/m2/full/*.png'

In [ ]:
import glob, os, shutil
REP = os.path.join("reports/m11/full", TAG)
def pack():
    stage = "/kaggle/working/m11_stage"
    shutil.rmtree(stage, ignore_errors=True)
    os.makedirs(stage)    # an empty zip still comes back when a step wrote nothing
    if os.path.isdir(REP):
        shutil.copytree(REP, f"{stage}/{REP}")
    for r in sorted(glob.glob("/kaggle/working/results/runs/*_m11_verify_full")):
        shutil.copytree(r, f"{stage}/results/runs/{os.path.basename(r)}")
    shutil.make_archive(f"/kaggle/working/m11_full_seed{SEED}_outputs", "zip", stage)
    shutil.rmtree(stage)
    print("zip written:", sorted(os.listdir(REP)) if os.path.isdir(REP) else [])

In [ ]:
# fit (CPU, 4 threads as in M9 so the refit can match M9's scores), then llm (GPU), each in its own process
for step in ["fit", "llm"]:
    print("=====", step, flush=True)
    !python -X faulthandler scripts/m11_verify.py --env kaggle --pool full --set resources.n_jobs=4 seed={SEED} {LLM_SET} {TAG_ARG} --only {step}; echo "exit code: $?"
    pack()
    if not os.path.exists(f"{REP}/evidence.parquet"):
        break    # nothing to verify
if os.path.exists(f"{REP}/VERIFY.md"):
    print(open(f"{REP}/VERIFY.md", encoding="utf-8").read()[:8000])